In [17]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree
import sys
sys.path.append("..")
from error_func import error

case_name = "case1"
mesh_name = "coil_box"
solver1 = "moose"
solver2 = "ngsolve"

sol_moose = np.load(f"../../output/{case_name}/gauss/{case_name}_{solver1}.npy")
sol_ngsolve = np.load(f"../../output/{case_name}/gauss/{case_name}_{solver2}.npy")

print(sol_moose.shape)
print(sol_ngsolve.shape)

print(sol_moose.shape)
print(sol_ngsolve.shape)


(69629, 7)
(69629, 7)
(69629, 7)
(69629, 7)


In [18]:
print("Min and max X coord:")
print(np.min(sol_moose[:, 0:3]))
print(np.max(sol_moose[:, 0:3]))

print("Min and max Y coord:")
print(np.min(sol_moose[:, 0:3]))
print(np.max(sol_moose[:, 0:3]))

print("Min and max Z coord:")
print(np.min(sol_moose[:, 0:3]))
print(np.max(sol_moose[:, 0:3]))

print("")

print("Min and max X coord:")
print(np.min(sol_ngsolve[:, 0:3]))
print(np.max(sol_ngsolve[:, 0:3]))

print("Min and max Y coord:")
print(np.min(sol_ngsolve[:, 0:3]))
print(np.max(sol_ngsolve[:, 0:3]))

print("Min and max Z coord:")
print(np.min(sol_ngsolve[:, 0:3]))
print(np.max(sol_ngsolve[:, 0:3]))

Min and max X coord:
-0.099551605255125
0.09795298646876
Min and max Y coord:
-0.099551605255125
0.09795298646876
Min and max Z coord:
-0.099551605255125
0.09795298646876

Min and max X coord:
-0.099551605255125
0.09795298646876001
Min and max Y coord:
-0.099551605255125
0.09795298646876001
Min and max Z coord:
-0.099551605255125
0.09795298646876001


In [19]:
tree = KDTree(sol_moose[:, 0:3])
distances, indices = tree.query(sol_ngsolve[:, 0:3])

max_dist = np.max(distances)
print(f"Maximum alignment error (distance): {max_dist:.6e}")
if max_dist > 1e-4:
    print("Warning: Large distance detected. Are the geometries identical?")


sol_moose_reordered = sol_moose[indices, :]

np.save(f"../../output/{case_name}/gauss/{case_name}_{solver1}_reordered_to_{solver2}.npy", sol_moose_reordered)

Maximum alignment error (distance): 8.653332e-16


In [20]:
sol_moose = np.load(f"../../output/{case_name}/gauss/{case_name}_{solver1}_reordered_to_{solver2}.npy")
elec_pot_moose = sol_moose[:, 3]
curr_dens_moose = sol_moose[:, 4:7]
# mag_vec_moose = sol_moose[:, 3:6]

sol_ngsolve = np.load(f"../../output/{case_name}/gauss/{case_name}_{solver2}.npy")
elec_pot_ngsolve = sol_ngsolve[:, 3]
curr_dens_ngsolve = sol_ngsolve[:, 4:7]
# mag_vec_comsol = sol_comsol[:, 3:6]


print(elec_pot_moose.shape)
print(elec_pot_ngsolve.shape)
print(np.max(elec_pot_ngsolve))
print(np.max(elec_pot_moose))

print(np.min(elec_pot_ngsolve))
print(np.min(elec_pot_moose))

print("")

print(curr_dens_moose.shape)
print(curr_dens_ngsolve.shape)
print(np.max(curr_dens_moose))
print(np.max(curr_dens_ngsolve))
print(np.min(curr_dens_moose))
print(np.min(curr_dens_ngsolve))

(69629,)
(69629,)
311736.85662603955
311736.85662615
-2.4014742334932965e-14
0.0

(69629, 3)
(69629, 3)
34132254.212542
32770039.24039918
-32752864.413028
-32703522.9652724


In [21]:
mesh = sol_ngsolve.copy()

In [22]:
print(f"Coordinate errors between {solver1} and {solver2}:")

mesh = error(sol=sol_moose[:, 0:3], sol_ref=sol_ngsolve[:, 0:3], 
             eps = 1e-12, mesh=mesh, tag="vector", save_tag=None)

Coordinate errors between moose and ngsolve:

  * Max. absolute error in x direction  : 4.996e-16.
  * Avg. absolute error in x direction  : 1.147e-16.

  * Max. relative error in x direction : 4.995e-12 %.
  * Avg. relative error in x direction : 3.281e-13 %.

  * Max. absolute error in y direction  : 4.996e-16.
  * Avg. absolute error in y direction  : 1.028e-16.

  * Max. relative error in y direction : 4.992e-12 %.
  * Avg. relative error in y direction : 3.259e-13 %.

  * Max. absolute error in z direction  : 4.996e-16.
  * Avg. absolute error in z direction  : 7.845e-17.

  * Max. relative error in z direction : 4.996e-12 %.
  * Avg. relative error in z direction : 4.145e-13 %.


In [23]:
print(f"Current density errors between {solver1} and {solver2}:")

mesh = error(sol=curr_dens_moose, sol_ref=curr_dens_ngsolve, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag=None)

# print(curr_dens_moose[-1, :])
# print(curr_dens_ngsolve[-1, :])

Current density errors between moose and ngsolve:

  * Max. absolute error in x direction  : 3.296e+07.
  * Avg. absolute error in x direction  : 1.173e+06.

  * Max. relative error in x direction : 1.231e-05 %.
  * Avg. relative error in x direction : 2.133e-08 %.

  * Max. absolute error in y direction  : 3.389e+07.
  * Avg. absolute error in y direction  : 1.602e+06.

  * Max. relative error in y direction : 4.613e-09 %.
  * Avg. relative error in y direction : 5.632e-11 %.

  * Max. absolute error in z direction  : 3.413e+07.
  * Avg. absolute error in z direction  : 6.460e+05.

  * Max. relative error in z direction : 2.108e-05 %.
  * Avg. relative error in z direction : 2.406e-08 %.


In [24]:
print(f"Electric potential errors between {solver1} and {solver2}:")

mesh = error(sol=elec_pot_moose, sol_ref=elec_pot_ngsolve, 
             eps = 1e-6, mesh=mesh, tag="scalar", save_tag=None)

Electric potential errors between moose and ngsolve:

  * Max. absolute error : 1.362e-07.
  * Avg. absolute error : 2.258e-08.

  * Max. relative error : 8.060e-11 %.
  * Avg. relative error : 6.317e-11 %.
